In [ ]:
import pandas as pd
import numpy as np

from deep_translator import GoogleTranslator
from deep_translator.exceptions import TranslationNotFound




import deepl 
translator = deepl.Translator("your-api-key")


from easynmt import EasyNMT
import nltk
import time
model = EasyNMT('opus-mt')

In [ ]:
trans = pd.read_csv('path_to_GSA_to_be_translated.csv') # Update with your file path
lang = 'da' # Update with the appropriate language code (e.g., 'de' for German, 'fr' for French, etc.)

In [ ]:
def g_translate(text, lang):
    if pd.isna(text) or str(text).strip() == "":
        return None
    try:
        return GoogleTranslator(source=lang, target='en').translate(text)
    except TranslationNotFound:
        return None
    except Exception:
        return None
    



def dpl_translate(text):
    if pd.isna(text) or str(text).strip() == "":
        return None

    for i in range(5):  # retry up to 5 times
        try:
            return translator.translate_text(text, target_lang="EN-US").text
        except deepl.exceptions.TooManyRequestsException:
            time.sleep(5 * (i + 1))  # increasing delay
        except Exception:
            return None
    return None    

In [ ]:
trans['g_tname'] = trans['original_name'].apply(lambda x: g_translate(x, lang))

In [ ]:

trans['dpl_tname'] = trans['original_name'].apply(dpl_translate)

In [ ]:

trans['opus_tname'] = trans.apply(lambda row:model.translate(row['original_name'],source_lang= lang ,target_lang='en') ,axis=1)


In [ ]:
# matching translation results from all three methods
trans["translation_flag"] = (
    (trans["g_tname"] == trans["dpl_tname"]) & 
    (trans["dpl_tname"] == trans["opus_tname"])
).astype(int)

In [ ]:
trans.to_csv('path_to_save_translated_GSA.csv', index=False)